# Data Preparation for GWM-RNN Relation Prediction - WN18-RR

This notebook prepares the WN18-RR dataset for training GWM-RNN on the Knowledge Graph Completion task using provided entity and relation text mappings.

## Dataset Overview
- **WN18-RR**: WordNet-based knowledge graph (cleaned version of WN18)
- **Entities**: Synsets (WordNet offsets as IDs)
- **Relations**: 11 relation types (hypernym, hyponym, meronym, holonym, similar_to, etc.)
- **Triples**: Training, validation, and test splits
- **Text mappings**: `entity2text.txt` and `relation2text.txt`
- **Task**: Given `(head, relation, ?)`, predict the tail entity

## Processing Steps
1. Load raw triples from TSV files
2. Load entity/relation text mappings
3. Create entity and relation vocabularies
4. Generate inverse relations (doubles the data)
5. Encode entity/relation descriptions using Sentence-BERT
6. Save processed tensors for training

## 1. Setup and Configuration

In [24]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from collections import defaultdict
import json
import torch

# Paths
RAW_DATA_DIR = Path(r"D:\NLP research\Code\graph-world-models\GWM\gwm-rnn\data\wn18-rr\raw\ID-ed")
OUTPUT_DIR = Path(r"D:\NLP research\Code\graph-world-models\GWM\gwm-rnn\data\wn18-rr\processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Raw file names
TRAIN_FILE = "train.tsv"
VALID_FILE = "dev.tsv"
TEST_FILE = "test.tsv"
ENTITY_TEXT_FILE = "entity2text.txt"
RELATION_TEXT_FILE = "relation2text.txt"

# Model configuration
EMBEDDING_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
BATCH_SIZE = 256
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
CREATE_INVERSE_RELATIONS = True

print(f"Raw data directory: {RAW_DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Train file: {TRAIN_FILE}")
print(f"Valid file: {VALID_FILE}")
print(f"Test file: {TEST_FILE}")
print(f"Entity text file: {ENTITY_TEXT_FILE}")
print(f"Relation text file: {RELATION_TEXT_FILE}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Device: {DEVICE}")
print(f"Create inverse relations: {CREATE_INVERSE_RELATIONS}")

Raw data directory: D:\NLP research\Code\graph-world-models\GWM\gwm-rnn\data\wn18-rr\raw\ID-ed
Output directory: D:\NLP research\Code\graph-world-models\GWM\gwm-rnn\data\wn18-rr\processed
Train file: train.tsv
Valid file: dev.tsv
Test file: test.tsv
Entity text file: entity2text.txt
Relation text file: relation2text.txt
Embedding model: sentence-transformers/all-MiniLM-L6-v2
Device: cpu
Create inverse relations: True


## 2. Load Entity and Relation Text Mappings

In [25]:
def load_text_mapping(file_path):
    """Load tab-separated text mappings: id<TAB>description."""
    mapping = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if '\t' in line:
                key, text = line.split('\t', 1)
            else:
                parts = line.split(None, 1)
                if len(parts) != 2:
                    continue
                key, text = parts
            mapping[key] = text
    return mapping

entity_texts_map = load_text_mapping(RAW_DATA_DIR / ENTITY_TEXT_FILE)
relation_texts_map = load_text_mapping(RAW_DATA_DIR / RELATION_TEXT_FILE)

print(f"Entity text mappings: {len(entity_texts_map):,}")
print(f"Relation text mappings: {len(relation_texts_map):,}")
print("\nExample entity mapping:")
for i, (k, v) in enumerate(entity_texts_map.items()):
    print(f"  {k} -> {v}")
    if i >= 2:
        break
print("\nExample relation mapping:")
for i, (k, v) in enumerate(relation_texts_map.items()):
    print(f"  {k} -> {v}")
    if i >= 2:
        break

Entity text mappings: 40,943
Relation text mappings: 11

Example entity mapping:
  14854262 -> stool, solid excretory product evacuated from the bowels
  00590383 -> chieftainship, the position of chieftain
  08769179 -> saxony, an area in Germany around the upper Elbe river; the original home of the Saxons

Example relation mapping:
  _member_of_domain_usage -> member of domain usage
  _has_part -> has part
  _also_see -> also see


## 3. Load Raw Data

In [26]:
def load_triples(file_path):
    """Load triples from tab-separated file."""
    triples = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) == 3:
                h, r, t = parts
                triples.append((h, r, t))
    return triples

# Load all splits
train_triples = load_triples(RAW_DATA_DIR / TRAIN_FILE)
valid_triples = load_triples(RAW_DATA_DIR / VALID_FILE)
test_triples = load_triples(RAW_DATA_DIR / TEST_FILE)

print(f"Training triples: {len(train_triples):,}")
print(f"Validation triples: {len(valid_triples):,}")
print(f"Test triples: {len(test_triples):,}")
print(f"Total triples: {len(train_triples) + len(valid_triples) + len(test_triples):,}")

# Show examples
print("\nExample triples:")
for i, triple in enumerate(train_triples[:5], 1):
    print(f"{i}. {triple}")

Training triples: 86,835
Validation triples: 3,034
Test triples: 3,134
Total triples: 93,003

Example triples:
1. ('00260881', '_hypernym', '00260622')
2. ('01332730', '_derivationally_related_form', '03122748')
3. ('06066555', '_derivationally_related_form', '00645415')
4. ('09322930', '_instance_hypernym', '09360122')
5. ('07193596', '_derivationally_related_form', '00784342')


## 4. Create Vocabularies and Inverse Relations

In [27]:
def create_vocabularies(train_triples, valid_triples, test_triples, create_inverse=True):
    """Create entity and relation vocabularies with optional inverse relations."""
    entities = set()
    relations = set()
    
    all_triples = train_triples + valid_triples + test_triples
    for h, r, t in all_triples:
        entities.add(h)
        entities.add(t)
        relations.add(r)
    
    if create_inverse:
        original_relations = list(relations)
        for rel in original_relations:
            relations.add(rel + '_inv')
    
    entity2id = {ent: idx for idx, ent in enumerate(sorted(entities))}
    id2entity = {idx: ent for ent, idx in entity2id.items()}
    relation2id = {rel: idx for idx, rel in enumerate(sorted(relations))}
    id2relation = {idx: rel for rel, idx in relation2id.items()}
    
    return entity2id, id2entity, relation2id, id2relation

entity2id, id2entity, relation2id, id2relation = create_vocabularies(
    train_triples, valid_triples, test_triples, 
    create_inverse=CREATE_INVERSE_RELATIONS
)

print(f"Number of entities: {len(entity2id):,}")
print(f"Number of relations: {len(relation2id):,}")

if CREATE_INVERSE_RELATIONS:
    original_rels = [r for r in relation2id.keys() if not r.endswith('_inv')]
    inverse_rels = [r for r in relation2id.keys() if r.endswith('_inv')]
    print(f"  - Original relations: {len(original_rels)}")
    print(f"  - Inverse relations: {len(inverse_rels)}")

with open(OUTPUT_DIR / 'entity2id.json', 'w') as f:
    json.dump(entity2id, f, indent=2)
with open(OUTPUT_DIR / 'relation2id.json', 'w') as f:
    json.dump(relation2id, f, indent=2)

print("\n✓ Vocabularies saved")

Number of entities: 40,943
Number of relations: 22
  - Original relations: 11
  - Inverse relations: 11

✓ Vocabularies saved


### Entity Count Verification

**Expected from literature (Dettmers et al. 2018)**: 40,943 entities

Let's verify the entity count and investigate any discrepancies:

In [28]:
# Count entities in each split
train_entities = set()
valid_entities = set()
test_entities = set()

for h, r, t in train_triples:
    train_entities.add(h)
    train_entities.add(t)

for h, r, t in valid_triples:
    valid_entities.add(h)
    valid_entities.add(t)

for h, r, t in test_triples:
    test_entities.add(h)
    test_entities.add(t)

all_entities = train_entities | valid_entities | test_entities

print("Entity Count Analysis:")
print("=" * 70)
print(f"Entities in train set: {len(train_entities):,}")
print(f"Entities in valid set: {len(valid_entities):,}")
print(f"Entities in test set: {len(test_entities):,}")
print(f"Total unique entities (union): {len(all_entities):,}")
print(f"\nFrom vocabulary: {len(entity2id):,}")
print(f"\n{'✓ Match' if len(all_entities) == len(entity2id) else '✗ Mismatch'}")

print(f"\nExpected from literature: 40,943")
print(f"Observed in our data: {len(entity2id):,}")
print(f"Difference: {len(entity2id) - 40943:+,}")

# Check for entities only in certain splits
train_only = train_entities - valid_entities - test_entities
valid_only = valid_entities - train_entities - test_entities
test_only = test_entities - train_entities - valid_entities

print(f"\nEntity Distribution:")
print(f"  Entities only in train: {len(train_only):,}")
print(f"  Entities only in valid: {len(valid_only):,}")
print(f"  Entities only in test: {len(test_only):,}")

# Check the raw file stats
print(f"\nRaw File Statistics:")
print(f"  Train triples: {len(train_triples):,}")
print(f"  Valid triples: {len(valid_triples):,}")
print(f"  Test triples: {len(test_triples):,}")

Entity Count Analysis:
Entities in train set: 40,559
Entities in valid set: 5,173
Entities in test set: 5,323
Total unique entities (union): 40,943

From vocabulary: 40,943

✓ Match

Expected from literature: 40,943
Observed in our data: 40,943
Difference: +0

Entity Distribution:
  Entities only in train: 31,473
  Entities only in valid: 175
  Entities only in test: 186

Raw File Statistics:
  Train triples: 86,835
  Valid triples: 3,034
  Test triples: 3,134


## 5. Create Entity and Relation Descriptions

In [29]:
def create_entity_texts(entity_list, text_mapping):
    """
    Create entity text from provided mappings with a safe fallback.
    """
    entity_texts = {}
    missing_count = 0

    for entity in entity_list:
        text = text_mapping.get(entity)
        if text:
            entity_texts[entity] = text
        else:
            missing_count += 1
            entity_texts[entity] = entity.replace('_', ' ')

    if missing_count > 0:
        print(f"⚠️  Warning: {missing_count:,} entities missing from entity2text.txt")

    return entity_texts

entity_texts = create_entity_texts(list(entity2id.keys()), entity_texts_map)

print(f"Entity texts created: {len(entity_texts):,}")
print("\nExample entity texts:")
for entity in list(entity2id.keys())[:8]:
    text = entity_texts[entity]
    print(f"  {entity:20s} -> {text}")

Entity texts created: 40,943

Example entity texts:
  00001740             -> take a breath, that which is perceived or known or inferred to have its own distinct existence (living or nonliving)   draw air into, and expel out of, the lungs; "I can breathe better when the air is clean"; "The patient is respiring"   (usually followed by `to') having the necessary means or skill or know-how or authority to do something; "able to swim"; "she was able to program her computer"; "we were at last able to buy a car"; "able to get a grant for the project"   without musical accompaniment; "they performed a cappella"
  00001930             -> physical entity, an entity that has physical existence
  00002137             -> abstraction, a general concept formed by extracting common features from specific examples
  00002325             -> respire, undergo the biomedical and metabolic processes of respiration by taking up oxygen and producing carbon monoxide
  00002452             -> thing, a separat

In [30]:
def create_relation_texts(relation2id, text_mapping):
    """
    Create relation text from provided mappings and add inverse descriptions.
    """
    relation_texts = {}

    for relation in relation2id.keys():
        if relation.endswith('_inv'):
            original = relation[:-4]
            base_text = text_mapping.get(original, original.replace('_', ' ').strip())
            text = f"inverse of {base_text}"
        else:
            text = text_mapping.get(relation, relation.replace('_', ' ').strip())

        relation_texts[relation] = text

    return relation_texts

relation_texts = create_relation_texts(relation2id, relation_texts_map)

print(f"Relation texts created: {len(relation_texts):,}")
print("\nRelation descriptions:")
for relation in sorted(relation_texts):
    print(f"  {relation:20s} -> {relation_texts[relation]}")

Relation texts created: 22

Relation descriptions:
  _also_see            -> also see
  _also_see_inv        -> inverse of also see
  _derivationally_related_form -> derivationally related form
  _derivationally_related_form_inv -> inverse of derivationally related form
  _has_part            -> has part
  _has_part_inv        -> inverse of has part
  _hypernym            -> hypernym
  _hypernym_inv        -> inverse of hypernym
  _instance_hypernym   -> instance hypernym
  _instance_hypernym_inv -> inverse of instance hypernym
  _member_meronym      -> member meronym
  _member_meronym_inv  -> inverse of member meronym
  _member_of_domain_region -> member of domain region
  _member_of_domain_region_inv -> inverse of member of domain region
  _member_of_domain_usage -> member of domain usage
  _member_of_domain_usage_inv -> inverse of member of domain usage
  _similar_to          -> similar to
  _similar_to_inv      -> inverse of similar to
  _synset_domain_topic_of -> synset domain top

In [31]:
def create_triple_texts(triples, entity_texts, relation_texts):
    """
    Create composite text for each triple: head + relation + tail.
    """
    triple_texts = []
    
    for h, r, t in triples:
        h_text = entity_texts[h]
        r_text = relation_texts[r]
        t_text = entity_texts[t]
        triple_text = f"{h_text} {r_text} {t_text}"
        triple_texts.append(triple_text)
    
    return triple_texts

train_triple_texts = create_triple_texts(train_triples, entity_texts, relation_texts)
valid_triple_texts = create_triple_texts(valid_triples, entity_texts, relation_texts)
test_triple_texts = create_triple_texts(test_triples, entity_texts, relation_texts)

print(f"Triple-level text attributes created:")
print(f"  Train: {len(train_triple_texts):,}")
print(f"  Valid: {len(valid_triple_texts):,}")
print(f"  Test: {len(test_triple_texts):,}")

print(f"\nExample triple descriptions:")
for i, triple_text in enumerate(train_triple_texts[:3], 1):
    text = triple_text
    if len(text) > 85:
        text = text[:82] + "..."
    print(f"{i}. {text}")

triple_texts_dict = {
    'train': train_triple_texts,
    'valid': valid_triple_texts,
    'test': test_triple_texts,
}
with open(OUTPUT_DIR / 'triple_texts.json', 'w') as f:
    json.dump(triple_texts_dict, f, indent=2)
print("\n✓ Triple texts saved to triple_texts.json")

Triple-level text attributes created:
  Train: 86,835
  Valid: 3,034
  Test: 3,134

Example triple descriptions:
1. land reform, a redistribution of agricultural land (especially by government actio...
2. cover, provide with a covering or cause to be covered; "cover her face with a hand...
3. phytology, the branch of biology that studies plants derivationally related form b...

✓ Triple texts saved to triple_texts.json


## 6. Generate Text Embeddings

Use Sentence-BERT to encode all entities and relations into dense vectors.

In [ ]:
try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    print("Installing sentence-transformers...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"])
    from sentence_transformers import SentenceTransformer

print(f"Loading embedding model: {EMBEDDING_MODEL}")
encoder = SentenceTransformer(EMBEDDING_MODEL)
encoder = encoder.to(DEVICE)

print(f"Model loaded on {DEVICE}")
print(f"Embedding dimension: {encoder.get_sentence_embedding_dimension()}")

In [ ]:
print("Encoding entities...")
entity_list = sorted(entity2id.keys(), key=lambda x: entity2id[x])
entity_text_list = [entity_texts[e] for e in entity_list]

entity_embeddings = encoder.encode(
    entity_text_list,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    device=DEVICE
)

print(f"Entity embeddings shape: {entity_embeddings.shape}")
print(f"Memory size: {entity_embeddings.nbytes / (1024**2):.2f} MB")

In [ ]:
print("Encoding relations...")
relation_list = sorted(relation2id.keys(), key=lambda x: relation2id[x])
relation_text_list = [relation_texts[r] for r in relation_list]

relation_embeddings = encoder.encode(
    relation_text_list,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    convert_to_numpy=True,
    device=DEVICE
)

print(f"Relation embeddings shape: {relation_embeddings.shape}")
print(f"Memory size: {relation_embeddings.nbytes / (1024**2):.2f} MB")

## 7. Convert Triples to Tensor Format

In [ ]:
def convert_triples_to_ids(triples, entity2id, relation2id, add_inverse=True):
    """Convert triples from strings to IDs, optionally adding inverse triples."""
    id_triples = []
    
    for h, r, t in triples:
        h_id = entity2id[h]
        r_id = relation2id[r]
        t_id = entity2id[t]
        id_triples.append((h_id, r_id, t_id))
        
        if add_inverse:
            r_inv_id = relation2id[r + '_inv']
            id_triples.append((t_id, r_inv_id, h_id))
    
    return np.array(id_triples, dtype=np.int32)

train_ids = convert_triples_to_ids(train_triples, entity2id, relation2id, CREATE_INVERSE_RELATIONS)
valid_ids = convert_triples_to_ids(valid_triples, entity2id, relation2id, CREATE_INVERSE_RELATIONS)
test_ids = convert_triples_to_ids(test_triples, entity2id, relation2id, CREATE_INVERSE_RELATIONS)

print(f"Training triples (with inverses): {len(train_ids):,}")
print(f"Validation triples (with inverses): {len(valid_ids):,}")
print(f"Test triples (with inverses): {len(test_ids):,}")
print(f"\nTotal processed triples: {len(train_ids) + len(valid_ids) + len(test_ids):,}")

if CREATE_INVERSE_RELATIONS:
    original_total = len(train_triples) + len(valid_triples) + len(test_triples)
    processed_total = len(train_ids) + len(valid_ids) + len(test_ids)
    print(f"Data augmentation: {original_total:,} -> {processed_total:,} ({processed_total/original_total:.1f}x)")

## 8. Create Ground Truth Dictionary

In [ ]:
def create_ground_truth_dict(train_ids, valid_ids, test_ids):
    """Create mapping (h, r) -> valid tails for filtered evaluation."""
    ground_truth = defaultdict(set)
    
    all_triples = np.concatenate([train_ids, valid_ids, test_ids], axis=0)
    
    for h, r, t in all_triples:
        ground_truth[(h, r)].add(t)
    
    ground_truth = {k: list(v) for k, v in ground_truth.items()}
    
    return ground_truth

ground_truth = create_ground_truth_dict(train_ids, valid_ids, test_ids)

print(f"Ground truth entries: {len(ground_truth):,}")
print(f"Average tails per (h, r): {np.mean([len(v) for v in ground_truth.values()]):.2f}")

ground_truth_json = {f"{h},{r}": [int(t) for t in tails] for (h, r), tails in ground_truth.items()}
with open(OUTPUT_DIR / 'ground_truth.json', 'w') as f:
    json.dump(ground_truth_json, f)

print("✓ Ground truth saved")

## 9. Save All Processed Data

In [ ]:
torch.save(torch.from_numpy(entity_embeddings), OUTPUT_DIR / 'entity_embeddings.pt')
torch.save(torch.from_numpy(relation_embeddings), OUTPUT_DIR / 'relation_embeddings.pt')

torch.save(torch.from_numpy(train_ids), OUTPUT_DIR / 'train_triples.pt')
torch.save(torch.from_numpy(valid_ids), OUTPUT_DIR / 'valid_triples.pt')
torch.save(torch.from_numpy(test_ids), OUTPUT_DIR / 'test_triples.pt')

entity_texts_dict = {ent: text for ent, text in sorted(entity_texts.items())}
relation_texts_dict = {rel: text for rel, text in sorted(relation_texts.items())}

with open(OUTPUT_DIR / 'entity_texts.json', 'w') as f:
    json.dump(entity_texts_dict, f, indent=2)
with open(OUTPUT_DIR / 'relation_texts.json', 'w') as f:
    json.dump(relation_texts_dict, f, indent=2)

metadata = {
    'dataset': 'WN18-RR',
    'description': 'WordNet-based Knowledge Graph with provided text mappings',
    'num_entities': len(entity2id),
    'num_relations': len(relation2id),
    'num_original_relations': len([r for r in relation2id if not r.endswith('_inv')]),
    'embedding_dim': entity_embeddings.shape[1],
    'embedding_model': EMBEDDING_MODEL,
    'text_enhancement': 'Entity/relation descriptions from entity2text.txt and relation2text.txt',
    'has_inverse_relations': CREATE_INVERSE_RELATIONS,
    'train_size': len(train_ids),
    'valid_size': len(valid_ids),
    'test_size': len(test_ids),
}

with open(OUTPUT_DIR / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print("="*70)
print("DATA PREPARATION COMPLETE WITH TEXT MAPPINGS")
print("="*70)
print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nFiles saved:")
print(f"  • entity_embeddings.pt {entity_embeddings.shape}")
print(f"  • relation_embeddings.pt {relation_embeddings.shape}")
print(f"  • train_triples.pt ({len(train_ids):,} triples)")
print(f"  • valid_triples.pt ({len(valid_ids):,} triples)")
print(f"  • test_triples.pt ({len(test_ids):,} triples)")
print(f"  • entity2id.json, relation2id.json")
print(f"  • entity_texts.json (provided descriptions)")
print(f"  • relation_texts.json (provided descriptions)")
print(f"  • triple_texts.json (composed triple descriptions)")
print(f"  • ground_truth.json")
print(f"  • metadata.json")
print(f"\nDataset statistics:")
for key, value in metadata.items():
    if key not in ['description', 'text_enhancement']:
        print(f"  • {key}: {value}")
print("\n✓ Ready for training with text-mapped entity/relation context!")

## 10. Create context embeddings

In [ ]:
required_files = ['generate_context_embeddings.py']

print("="*70)
print("Cloning GitHub repository...")
print("="*70)

# Clone your GitHub repo
GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
BRANCH = "context-aware-gwm-rnn"

!git clone {GITHUB_REPO} /kaggle/working/gwm
%cd /kaggle/working/gwm
!git checkout {BRANCH}
!git pull
%cd ../

# Copy files from repo to working directory
repo_path = "/kaggle/working/gwm/gwm-rnn/relation-prediction"

print(f"\nCopying files from {repo_path}...")
for file in required_files:
    !cp {repo_path}/{file} /kaggle/working/
    print(f"✓ Copied {file}")

# Verify files exist
import os
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"\n❌ Missing files: {missing_files}")
    raise FileNotFoundError(f"Required files not found: {missing_files}")
else:
    print(f"\n✓ All required files ready: {required_files}")

In [ ]:
!python generate_context_embeddings.py \
    --data_dir /kaggle/working/dataset/ \
    --aggregation mean \

## 11. Data Statistics and Visualization

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

relation_counts = defaultdict(int)
for h, r, t in train_ids:
    relation_counts[r] += 1

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

top_relations = sorted(relation_counts.items(), key=lambda x: x[1], reverse=True)[:15]
rel_names = [id2relation[r] for r, _ in top_relations]
rel_counts = [c for _, c in top_relations]

axes[0].barh(range(len(rel_names)), rel_counts, color='steelblue')
axes[0].set_yticks(range(len(rel_names)))
axes[0].set_yticklabels(rel_names, fontsize=9)
axes[0].set_xlabel('Number of Triples', fontsize=11)
axes[0].set_title('Most Frequent Relations (Top 15)', fontsize=12, fontweight='bold')
axes[0].invert_yaxis()
axes[0].grid(axis='x', alpha=0.3)

counts = list(relation_counts.values())
axes[1].hist(counts, bins=20, edgecolor='black', color='steelblue', alpha=0.7)
axes[1].set_xlabel('Number of Triples', fontsize=11)
axes[1].set_ylabel('Number of Relations', fontsize=11)
axes[1].set_title('Distribution of Relation Frequencies', fontsize=12, fontweight='bold')
axes[1].set_yscale('log')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'relation_statistics.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Most frequent relation: {id2relation[top_relations[0][0]]} ({top_relations[0][1]} triples)")
print(f"Least frequent relation (in top 15): {id2relation[top_relations[-1][0]]} ({top_relations[-1][1]} triples)")
print(f"Average triples per relation: {np.mean(counts):.1f}")
print(f"Median triples per relation: {np.median(counts):.1f}")

## 11. Text Attribute Summary

In [ ]:
print("="*70)
print("TEXT ATTRIBUTE SUMMARY")
print("="*70)

print("\n📚 Entity Descriptions (from entity2text.txt):")
print("-" * 70)
sample_entities = list(entity2id.keys())[:6]
for entity in sample_entities:
    text = entity_texts[entity]
    if len(text) > 75:
        text = text[:72] + "..."
    print(f"  {entity:20s} -> {text}")

print("\n📖 Relation Mappings (from relation2text.txt):")
print("-" * 70)
original_rels = sorted([r for r in relation2id.keys() if not r.endswith('_inv')])
for rel in original_rels[:6]:
    print(f"  {rel:20s} -> {relation_texts[rel]}")
    print(f"  {rel}_inv{'':<13} -> {relation_texts[rel + '_inv']}")

print("\n🎯 Triple-Level Composite Texts:")
print("-" * 70)
for i, raw_triple in enumerate(train_triples[:3], 1):
    h, r, t = raw_triple
    composite = f"{entity_texts[h]} {relation_texts[r]} {entity_texts[t]}"
    if len(composite) > 75:
        composite = composite[:72] + "..."
    print(f"  {i}. {composite}")

print("\n✨ Text Mappings Applied:")
print("  ✓ Entity descriptions: entity2text.txt")
print("  ✓ Relation texts: relation2text.txt")
print("  ✓ Triple composition: head + relation + tail context")
print("  ✓ Inverse relations: 'inverse of' prefix")
print("  ✓ Fallbacks: Missing entities use cleaned IDs")
print("\n" + "="*70)